In [5]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import median_absolute_error
from sklearn.tree import DecisionTreeRegressor


def print_metrics(name, y_test, y_pred):
  mae = mean_absolute_error(y_test, y_pred)
  mse = mean_squared_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)
  mape = mean_absolute_percentage_error(y_test, y_pred)
  med_ae = median_absolute_error(y_test, y_pred)

  print(f"\nACCURACY USING {name}")
  print(f"R2 Score (Fit):         {r2:.4f}")
  print(f"MAE (Avg Error):        {mae:.2f} FPS")
  print(f"Median AE (Typical):    {med_ae:.2f} FPS")
  print(f"RMSE (Penalty Error):   {rmse:.2f} FPS")
  print(f"MAPE (Relative Error):  {mape*100:.2f}%\n")

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline

# Load the dataset
train_df = pd.read_csv("FPS_Combinedx6.csv", sep=";")
hidden_df = pd.read_csv("FPS_hidden.csv", sep=";")

# Define the features and the target
features = ['Quality', 'Polygons', 'Lights', 'Shadows', 'Particles', 'IsComplexMaterial']
target = 'FPS'

X_train = train_df[features]
y_train = train_df[target]
X_hidden = hidden_df[features]
y_hidden = hidden_df[target]

# Creates a group ID based on the unique combination of inputs.
train_df['group_id'] = train_df.groupby(features).ngroup()
groups = train_df['group_id']

# CROSS-VALIDATION SETUP
gkf = GroupKFold(n_splits=5)

# Dictionary to store performance metrics for final comparison
model_performance = {}


In [7]:
from sklearn.linear_model import LinearRegression

#  BASE MODEL - Linear Regression

print("\nBase Model: Linear Regression \n")

# Feature space lifting: include interactions (Xj * Xk) and polynomial terms (Xj^2)
# per the rule: keeping the original features is handled by include_bias=False/interaction_only=False
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lift', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])

# Evaluate Base Model via Cross-Validation
cv_r2_scores = []
for train_idx, val_idx in gkf.split(X_train, y_train, groups=groups):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    base_pipeline.fit(X_tr, y_tr)
    preds = base_pipeline.predict(X_val)
    cv_r2_scores.append(r2_score(y_val, preds))

# Fit on full training data
base_pipeline.fit(X_train, y_train)
base_hidden_preds = base_pipeline.predict(X_hidden)

print_metrics("Base Lifted Linear Regression", y_hidden, base_hidden_preds)


Base Model: Linear Regression 


ACCURACY USING Base Lifted Linear Regression
R2 Score (Fit):         0.4286
MAE (Avg Error):        18.20 FPS
Median AE (Typical):    13.26 FPS
RMSE (Penalty Error):   25.04 FPS
MAPE (Relative Error):  247.78%



In [8]:
from sklearn.ensemble import RandomForestRegressor

#  ADVANCED 1 - Random Forest Regressor (with Hyperparameters)

print("\nAdvanced Model 1: Random Forest Regressor \n")

rf_pipeline = Pipeline([
    ('rf', RandomForestRegressor(random_state=42))
])

# Hyperparameters to tune
rf_param_grid = {
    'rf__n_estimators': [100, 300, 600],
    'rf__max_depth': [10, 20, None],
    'rf__min_samples_leaf': [1, 4, 16, 32, 64],
    'rf__max_features': ['sqrt', 1.0]
}

# GridSearchCV using GroupKFold to keep hyperparameter selection untainted
rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    cv=gkf,
    scoring='r2',
    n_jobs=-1
)

rf_search.fit(X_train, y_train, groups=groups)
best_rf_model = rf_search.best_estimator_

rf_hidden_preds = best_rf_model.predict(X_hidden)

print_metrics("Advanced Random Forest", y_hidden, rf_hidden_preds)
print(f"Best RF Parameters: {rf_search.best_params_}\n")


Advanced Model 1: Random Forest Regressor 


ACCURACY USING Advanced Random Forest
R2 Score (Fit):         0.8400
MAE (Avg Error):        7.73 FPS
Median AE (Typical):    3.64 FPS
RMSE (Penalty Error):   13.25 FPS
MAPE (Relative Error):  48.29%

Best RF Parameters: {'rf__max_depth': 10, 'rf__max_features': 1.0, 'rf__min_samples_leaf': 4, 'rf__n_estimators': 300}



In [9]:
print("\nAdvanced Model 3: Decision Tree \n")

dt_pipeline = Pipeline([
    ('dt', DecisionTreeRegressor(random_state=42))
])

dt_param_grid = {
    'dt__criterion': ['squared_error', 'friedman_mse', 'absolute_error'],
    'dt__max_depth': [5, 10,15, 20, None],
    'dt__min_samples_split': [2, 5, 10],
    'dt__min_samples_leaf': [1, 2, 4],
    'dt__max_features': [None, 'sqrt', 'log2']
}

dt_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

dt_search.fit(X_train, y_train, groups=groups)
best_dt_model = dt_search.best_estimator_

dt_hidden_preds = best_dt_model.predict(X_hidden)

print_metrics("Advanced Decision Tree", y_hidden, dt_hidden_preds)
print(f"Best RF Parameters: {dt_search.best_params_}\n")




Advanced Model 3: Decision Tree 

Fitting 5 folds for each of 405 candidates, totalling 2025 fits


C:\Users\theok\Desktop\FPS_pred\.venv\Lib\site-packages\sklearn\model_selection\_split.py:87: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(



ACCURACY USING Advanced Decision Tree
R2 Score (Fit):         0.8236
MAE (Avg Error):        8.26 FPS
Median AE (Typical):    3.94 FPS
RMSE (Penalty Error):   13.92 FPS
MAPE (Relative Error):  49.83%

Best RF Parameters: {'dt__criterion': 'absolute_error', 'dt__max_depth': 10, 'dt__max_features': None, 'dt__min_samples_leaf': 1, 'dt__min_samples_split': 5}

